In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data_dir = "/content/drive/MyDrive/Dataset/internship/archive"

In [ ]:
# @title
for epoch in range(1):   # শুধু 1 epoch
    total_loss = 0
    for i, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if i % 20 == 0:
            print(f"Batch {i}, Loss: {loss.item()}")

    print(f"Epoch done. Total Loss: {total_loss}")

Batch 0, Loss: 0.666114866733551
Batch 20, Loss: 0.6594910025596619
Batch 40, Loss: 0.658397376537323


KeyboardInterrupt: 

In [ ]:
# Faster Training Version

for epoch in range(2):   # শুধু 2 epoch
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Epoch {epoch+1} | Loss: {total_loss:.4f} | Accuracy: {accuracy:.2f}%")

Epoch 1 | Loss: 91.5983 | Accuracy: 50.10%
Epoch 2 | Loss: 91.0603 | Accuracy: 49.51%


In [ ]:
torch.save(model, "model.pth")
print("Model saved successfully!")

Model saved successfully!


In [ ]:
import torch
torch.cuda.is_available()

True

In [ ]:
len(dataset)

4084

**🔴_______________________________
 ____________________________________________**

In [ ]:
# @title
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install scikit-learn tqdm

Looking in indexes: https://download.pytorch.org/whl/cu118


In [ ]:
# @title
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os

# Transform
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

# Dataset path (Google Drive এ upload করো dataset)
data_dir = "/content/drive/MyDrive/Dataset/internship/archive"

dataset = datasets.ImageFolder(data_dir, transform=transform)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# Load pretrained model
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train
for epoch in range(3):
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1} done")

# Save model
torch.save(model, "model.pth")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 160MB/s]


KeyboardInterrupt: 

In [ ]:
# @title
from pathlib import Path
import random, shutil

src_real = Path(f"{base}/wfir/real")
src_fake = Path(f"{base}/wfir/fake")

def make_split(src_dir, dst_root, seed=42, train_ratio=0.8, val_ratio=0.1):
    files = list(src_dir.glob("*"))
    random.Random(seed).shuffle(files)
    n = len(files)
    n_train = int(n*train_ratio)
    n_val = int(n*val_ratio)

    splits = {
        "train": files[:n_train],
        "val": files[n_train:n_train+n_val],
        "test": files[n_train+n_val:]
    }
    for split, fs in splits.items():
        d = Path(dst_root)/split/src_dir.name  # .../train/real or .../train/fake
        d.mkdir(parents=True, exist_ok=True)
        for f in fs:
            shutil.copy(f, d/f.name)

dst_root = f"{base}/dataset"
make_split(src_real, dst_root)
make_split(src_fake, dst_root)

!find {dst_root} -maxdepth 2 -type d -print

/content/drive/MyDrive/Dataset/internship/archive/real_and_fake_face_detection/dataset
/content/drive/MyDrive/Dataset/internship/archive/real_and_fake_face_detection/dataset/train
/content/drive/MyDrive/Dataset/internship/archive/real_and_fake_face_detection/dataset/train/real
/content/drive/MyDrive/Dataset/internship/archive/real_and_fake_face_detection/dataset/train/fake
/content/drive/MyDrive/Dataset/internship/archive/real_and_fake_face_detection/dataset/val
/content/drive/MyDrive/Dataset/internship/archive/real_and_fake_face_detection/dataset/val/real
/content/drive/MyDrive/Dataset/internship/archive/real_and_fake_face_detection/dataset/val/fake
/content/drive/MyDrive/Dataset/internship/archive/real_and_fake_face_detection/dataset/test
/content/drive/MyDrive/Dataset/internship/archive/real_and_fake_face_detection/dataset/test/real
/content/drive/MyDrive/Dataset/internship/archive/real_and_fake_face_detection/dataset/test/fake


In [ ]:
# @title
from sklearn.metrics import classification_report
import numpy as np

# Test evaluation
model.eval()
all_y, all_p = [], []
with torch.no_grad():
    for x,y in test_dl:
        x = x.to(device)
        out = model(x)
        preds = out.argmax(1).cpu().numpy()
        all_y.extend(y.numpy())
        all_p.extend(preds)
print(classification_report(all_y, all_p, target_names=class_names))

# Save weights + labels
save_dir = "/content/export"
os.makedirs(save_dir, exist_ok=True)
weights_path = f"{save_dir}/best_resnet18.pth"
torch.save({
    "state_dict": model.state_dict(),
    "class_names": class_names
}, weights_path)
print("Saved:", weights_path)

NameError: name 'test_dl' is not defined